# Code Notebooks and the TPR-DB

## Module 01: Lesson 01

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Critt-Kent/CRITT-Academy/blob/main/modules/01_Foundations/02_Notebooks_and_TPRDB.ipynb)

### What You Will Do In This Lesson (Steps)

1. Make sure your Jupyter kernel is set up
2. Install and import all necessary Python libraries
3. Learn how to use the `tprdb-utilities` function called `fetch_TPRDB_tables()`
4. Learn how to use the `tprdb-utilities` function called `read_TPRDB_tables()`
5. Look at the session data tables (SS tables) using a Pandas DataFrame object
6. Repeat Steps 4 and 5 with the segment (SG) tables (guided)
7. Repeat Steps 4 and 5 with the source-token (ST) tables (you explore 🤓🛩️)

## Step 1. Make sure your Jupyter kernel is running

Before executing any code in this notebook, please verify that your Python environment is active:

1. Look at the **top-right corner** of this window. 
2. Ensure it displays an active environment name or a version of Python (e.g., `Python 3.x`).
3. If it reads **Select Kernel** or **No Kernel**, click that text block and select the default **Python 3 (ipykernel)** or your preferred local environment from the dropdown menu.

*Once your kernel is connected (indicated by a solid or hollow circle next to the Python name in the top-right), you are ready for step 2.*

### Jupyter Basics

This first lesson is not meant to teach you how to use Jupyter notebooks, but here are some YouTube videos that might be helpful if you've never used a code notebook before:

- [Installation Guide and Basics](https://youtu.be/7oNFaX2q5uU?si=liiHpa_noI3_QQvo)
- [Common Mistakes and Essential Shortcuts](https://youtu.be/VJbL5LSqGDc?si=qG0bCjQaSlmsizRq)
- [Code Cells and Markdown Cells](https://youtu.be/Tlv94Z0lGpo?si=7v0eb5RHtF0IXnH0)

## Step 2. Install and import all necessary Python libraries

**Python libraries** are just packages of code that are used for a specific purpose. Here are the Python libraries that will be used in this very basic lesson:

- **Pandas**: `pandas` is used for data manipulation, organization, and analysis
- **tprdb-utilities**: `tprdb_utilities` is used for getting data remotely from the CRITT TPR-DB and then for reading data tables from a certain study (or studies) into a Pandas DataFrame.

### So what does the following code in step 2 do?

The code in this step is something you will see in all CRITT Academy lessons.

1. First, you make sure you have all the Python libraries installed that will be necessary for the lesson.
2. Then, you *import* the Python libraries so they are ready to use for the entire rest of the notebook


In [ ]:
# Environment Setup
import sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

# Enforce a minimum Python version of 3.9+
if sys.version_info < (3, 9):
    raise RuntimeError(
        f"❌ Python 3.9 or higher is required. "
        f"You are currently running Python {sys.version_info.major}.{sys.version_info.minor}."
    )

REQUIRED = {
    "pandas": "3.0.0",
    "tprdb-utilities": "2.0.0",
}

def _check_versions():
    outdated = []
    for pkg, min_ver in REQUIRED.items():
        try:
            installed = version(pkg)
            if Version(installed) < Version(min_ver):
                outdated.append(f"  {pkg}: installed {installed}, need >={min_ver}")
        except PackageNotFoundError:
            raise ImportError(f"{pkg} is not installed")
    return outdated

# Check if dependencies are missing and install them automatically
try:
    import numpy as np
    import pandas as pd
    import scipy.stats
    import matplotlib as mpl
    import seaborn as sns
    import tprdb_utilities as tpr

    outdated = _check_versions()
    if outdated:
        raise ImportError("Outdated packages:\n" + "\n".join(outdated))
    else:
        print("🤠 All core dependencies are already installed. You are ready to go! 🤘")
except ImportError:
    print("Missing dependencies detected. Installing required packages...")

    try:
        # The %pip magic ensures installation happens in the active Jupyter kernel
        %pip install "pandas>=3.0.0" "tprdb-utilities>=2.0.0"

        print("\n🤠 Installation complete 🤘 If imports fail on the next cell, please restart the kernel.")
    except Exception as e:
        print(f"❌ An error occurred during installation: {e}")
        print("If using Google Colab, you may just have to restart your runtime now to use the newly installed packages.")

Missing dependencies detected. Installing required packages...
Note: you may need to restart the kernel to use updated packages.

🤠 Installation complete 🤘 Please restart the kernel if imports fail on the next cell.


In [ ]:
# Now, import the libraries

try:
    # Standard Python library imports
    import pandas as pd

    # TPR-DB utilities import
    from tprdb_utilities import fetch_TPRDB_tables, read_TPRDB_tables

except ImportError as e:
    print(f"❌ An error occurred during imports: {e}")
    print("Please ensure all dependencies are installed and the kernel is restarted if necessary.")

## Step 3. Fetch some data tables from the CRITT TPR-DB

Data in the CRITT TPR-DB is living on a server in Kent, Ohio. If you want to analyze that data, you first need to ask yourself where you are.

### **!!! ⚠️ Important ⚠️ !!!**

This step **depends on where you are running this notebook from**:

#### **On the CRITT Jupyter server?**

If you are using Jupyter from the CRITT Jupyter server, then you are already in the same place as the data and you can skip all of step 3 (this step) and move on to step 4 (the next step).

---

#### **Are you somewhere else?**

If you are **not** on the CRITT Jupyter server, then this step is for you. Since the data is on a server that you are not working from, you need to download it.

You will use the `fetch_TPRDB_tables()` function from the `tprdb-utilities` library to download some data tables to wherever you currently are (your own computer, another Jupyter server, Google Colab, etc.). If you'd like to read the full documentation for this library and its functions, you can find it on the [Python Package Index](https://pypi.org/project/tprdb-utilities/), but we'll explain it simply right here:

The `fetch_TPRDB_tables()` function is very simple:
1. First, you tell it
    1. `path`: **where to save** the downloaded data
    2. `StudyID`: what TPR-DB **study** you would like to download from
    3. `extensions`: the data table **extension(s)** (one or several) you want to download
    4. **whether or not** the study you're downloading from is **public** (this lesson will only cover downloading from a public study)
2. Then, the function
    1. creates a folder called `tprdb-mothership-clone` in the location you gave it
    2. saves the data tables in the mothership clone folder

##### Help Docs

Before actually using the `fetch_TPRDB_tables()` function, you can see the help documentation by using the `?` shorthand character after the function name.

In [8]:
fetch_TPRDB_tables?

Signature:
fetch_TPRDB_tables(
    path,
    StudyID,
    extension,
    public,
    username=None,
    token=None,
    verbose=0,
)
Docstring:
Download TPR-DB data tables from the CRITT TPR-DB API and save them
locally in a directory structure that mirrors the TPR-DB server layout.

Makes one HTTP GET request per extension to the CRITT TPR-DB REST API,
receives a ``.zip`` archive, and extracts its contents directly into the
appropriate ``Tables/`` subdirectory.  The resulting file structure is
identical to the layout expected by ``read_TPRDB_tables``, so the two
functions are designed to be used in sequence.

**File structure created**::

    <path>/
    └── tprdb-mothership-clone/
        ├── PUBLIC/                 ← public studies
        │   └── <StudyID>/
        │       ├── studySummary.xml
        │       └── Tables/
        │           ├── session1.<ext>
        │           └── ...
        └── <username>/             ← private studies (when public=False)
            └── <Study

##### The **mothership clone**

It is important to use the same `path` variable for your "mothership clone" every time you use the `read_TPRDB_tables()` function because it will recognize if the data you want has already been downloaded and will conserve resources by not downloading it twice. It will also recognize if the CRITT TPR-DB has a more up-to-date version of the data tables than you already have in your mothership clone and will replace them with the more up-to-date version.

###### **!!! ⚠️ Important ⚠️ !!!**
Change the value of the `mothership_clone_location variable` below, and then we will be ready to run the `read_TPRDB_tables()` function

In [6]:
# Change what is in the quotes to point to where you want your mothership clone to be saved.
mothership_clone_location = "./"

In [ ]:
# fetch data tables (notice we are using the `mothership_clone_location` variable we defined in the cell above)

fetch_TPRDB_tables(
    path=mothership_clone_location,
    StudyID="SG12",
    extension=["ss", "sg", "st"],
    public=True,
)

=== fetch_TPRDB_tables Summary ===
StudyID  : SG12
Clone dir: ./tprdb-mothership-clone
User dir : PUBLIC

Extension  Status            Time
---------  ----------------  ------
ss         Up to date (304)  0.18s
sg         Up to date (304)  0.19s
st         Up to date (304)  0.16s

To read these files with read_TPRDB_tables:
  path      = "./tprdb-mothership-clone"
  user      = "PUBLIC"
  studies   = ["SG12"]


## Step 4: Read the tables into a DataFrame

Now you will use the `read_TPRDB_tables()` function to read a certain data table from one study into a DataFrame so that it can be analyzed.

### Help Docs

Once again, before you run the function, look at its help documentation:

In [ ]:
read_TPRDB_tables?

You will now read the SS tables (session tables) into a DataFrame using the `read_TPRDB_tables()` function. You will assign the output of the function to a variable called `sessions`, so `sessions` is now the name of the DataFrame containing the SS tables.

---

### **!!! ⚠️ Important ⚠️ !!!:**

Once again, where we are running this notebook from matters:

#### **On the CRITT Jupyter server?**

If this is the case, then you should copy and paste the following code into the next cell:

```py
sessions = read_TPRDB_tables(
    studies=["SG12"],
    extension="ss",
    mothership=True,
    user="PUBLIC",
    verbose=1,
)
```

---

#### **Are you somewhere else?**

In this case, you need to use your mothership clone you created before, so you should copy and paste the following code into the next cell (notice that we define the `path` of the mothership clone folder using the `mothership_clone_location` variable we declared in Step 3):

```py
sessions = read_TPRDB_tables(
    studies=["SG12"],
    extension="ss",
    mothership=False,
    path=f"{mothership_clone_location}/tprdb-mothership-clone",
    user="PUBLIC",
    verbose=1,
)
```

In [ ]:
# 💻 Paste correct code from previous markdown cell depending on your situation, and then run the cell.


Reading: SG12	with 139 'ss' Tables
Total 'ss' data rows: 139, columns: 29


## Step 5: Look at the DataFrame

You will now use some basic code just to look at the `sessions` DataFrame.

### 5.1: Ask for the DataFrame's shape

Whenever you make a DataFrame, it's a good idea to find out how big it is. In the previous step, if you had `verbose=1` for the `read_TPRDB_tables()` function, then you already saw the number of columns and rows, but you should know how to find the number of columns and rows (often referred to as the **"shape"**) for any DataFrame.



In [ ]:
# simply returns a tuple with no. rows and columns
print(sessions.shape)

# Fancy print statement to make it more readable 🤠
print(f"📊 The sessions table has {sessions.shape[0]} rows and {sessions.shape[1]} columns")

(139, 29)
📊 The sessions table has 139 rows and 29 columns


### 5.2: Hyper simple display

Simply running `sessions` (invoking the name of the DataFrame) will display the header of the DataFrame along with the first and last 5 rows (middle is truncated).

Notice that the very first column is unlabeled and is something that was not in the original data tables; this is the index for each row of the DataFrame. The index always starts at zero and counts up.

Also notice that this view shows a certain number of the first and last columns in the dataframe

In [10]:
sessions

,Id,Study,Session,StudySession,SL,TL,Task,Text,Part,KUI,...,TokS,LenS,TokT,LenT,FixS,TrtS,FixT,TrtT,HZ,FixRatio
0,1,SG12,P20_T2,SG12@P20_T2,en,de,T,2,P20,278,...,153,848,160,1018,1101,225999,1737,482519,300,0
1,1,SG12,P06_E3,SG12@P06_E3,en,de,E,3,P06,404,...,146,857,142,902,129,11109,3391,418214,300,0
2,1,SG12,P06_T6,SG12@P06_T6,en,de,T,6,P06,434,...,139,779,108,825,1689,244306,3755,521147,300,0
3,1,SG12,P04_P3,SG12@P04_P3,en,de,P,3,P04,310,...,146,857,154,970,1543,261513,2347,491944,300,0
4,1,SG12,P08_T5,SG12@P08_T5,en,de,T,5,P08,310,...,139,791,148,943,1380,313922,1383,377358,300,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,1,SG12,P06_E4,SG12@P06_E4,en,de,E,4,P06,436,...,110,668,88,668,105,56490,746,239931,300,0
135,1,SG12,P11_P3,SG12@P11_P3,en,de,P,3,P11,372,...,146,857,162,1011,614,138658,1594,454208,300,0
136,1,SG12,P24_P5,SG12@P24_P5,en,de,P,5,P24,280,...,139,791,153,990,993,129267,1646,220210,300,0
137,1,SG12,P24_E1,SG12@P24_E1,en,de,E,1,P24,278,...,160,838,168,1033,121,30324,1594,454866,300,0


### 5.3: First 5 rows

Running `sessions.head()` (invoking the `sessions` DataFrame object's `.head()` method) will show the first 5 rows. Columns are truncated in the same way as the previous example.

In [11]:
sessions.head()

,Id,Study,Session,StudySession,SL,TL,Task,Text,Part,KUI,...,TokS,LenS,TokT,LenT,FixS,TrtS,FixT,TrtT,HZ,FixRatio
0,1,SG12,P20_T2,SG12@P20_T2,en,de,T,2,P20,278,...,153,848,160,1018,1101,225999,1737,482519,300,0
1,1,SG12,P06_E3,SG12@P06_E3,en,de,E,3,P06,404,...,146,857,142,902,129,11109,3391,418214,300,0
2,1,SG12,P06_T6,SG12@P06_T6,en,de,T,6,P06,434,...,139,779,108,825,1689,244306,3755,521147,300,0
3,1,SG12,P04_P3,SG12@P04_P3,en,de,P,3,P04,310,...,146,857,154,970,1543,261513,2347,491944,300,0
4,1,SG12,P08_T5,SG12@P08_T5,en,de,T,5,P08,310,...,139,791,148,943,1380,313922,1383,377358,300,0


### 5.4: See without truncating the columns

Often, DataFrames have a limited, manageable number of **columns** but may have a *massive* number of **rows**. You will often want to override the Pandas default and see all columns.

In [19]:
with pd.option_context("display.max_columns", None):
    display(sessions.head())

,Id,Study,Session,StudySession,SL,TL,Task,Text,Part,KUI,PUB,DraftTime,RevTime,Dur,ALseg,STseg,TTseg,Ins,Del,TokS,LenS,TokT,LenT,FixS,TrtS,FixT,TrtT,HZ,FixRatio
0,1,SG12,P20_T2,SG12@P20_T2,en,de,T,2,P20,278,997,17909,614286,801066,7,7,7,1123,105,153,848,160,1018,1101,225999,1737,482519,300,0
1,1,SG12,P06_E3,SG12@P06_E3,en,de,E,3,P06,404,4407,23057,361174,576471,4,5,5,369,158,146,857,142,902,129,11109,3391,418214,300,0
2,1,SG12,P06_T6,SG12@P06_T6,en,de,T,6,P06,434,3394,8861,659978,1018094,7,7,7,991,86,139,779,108,825,1689,244306,3755,521147,300,0
3,1,SG12,P04_P3,SG12@P04_P3,en,de,P,3,P04,310,8383,127531,834216,918503,5,5,5,234,126,146,857,154,970,1543,261513,2347,491944,300,0
4,1,SG12,P08_T5,SG12@P08_T5,en,de,T,5,P08,310,1248,62790,730459,771238,6,6,8,1017,74,139,791,148,943,1380,313922,1383,377358,300,0


### 5.5: See the entire DataFrame

The following code can often be a less-than-efficient method when working with really large DataFrames (thousands or hundreds of thousands of rows), but since `sessions` is quite small, it's not too big a deal.

In [18]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(sessions)

,Id,Study,Session,StudySession,SL,TL,Task,Text,Part,KUI,PUB,DraftTime,RevTime,Dur,ALseg,STseg,TTseg,Ins,Del,TokS,LenS,TokT,LenT,FixS,TrtS,FixT,TrtT,HZ,FixRatio
0,1,SG12,P20_T2,SG12@P20_T2,en,de,T,2,P20,278,997,17909,614286,801066,7,7,7,1123,105,153,848,160,1018,1101,225999,1737,482519,300,0
1,1,SG12,P06_E3,SG12@P06_E3,en,de,E,3,P06,404,4407,23057,361174,576471,4,5,5,369,158,146,857,142,902,129,11109,3391,418214,300,0
2,1,SG12,P06_T6,SG12@P06_T6,en,de,T,6,P06,434,3394,8861,659978,1018094,7,7,7,991,86,139,779,108,825,1689,244306,3755,521147,300,0
3,1,SG12,P04_P3,SG12@P04_P3,en,de,P,3,P04,310,8383,127531,834216,918503,5,5,5,234,126,146,857,154,970,1543,261513,2347,491944,300,0
4,1,SG12,P08_T5,SG12@P08_T5,en,de,T,5,P08,310,1248,62790,730459,771238,6,6,8,1017,74,139,791,148,943,1380,313922,1383,377358,300,0
5,1,SG12,P13_E4,SG12@P13_E4,en,de,E,4,P13,310,6509,9610,415478,495475,5,5,5,151,81,110,668,103,716,141,19714,2521,410461,300,0
6,1,SG12,P02_P6,SG12@P02_P6,en,de,P,6,P02,310,4995,83788,887115,932121,7,7,7,258,189,139,779,130,855,1111,266836,2424,610644,300,0
7,1,SG12,P13_T1,SG12@P13_T1,en,de,T,1,P13,310,1556,19172,670508,892154,11,11,11,996,61,160,838,144,896,2864,325924,3501,358551,300,0
8,1,SG12,P24_E6,SG12@P24_E6,en,de,E,6,P24,310,4223,86503,386150,497394,7,7,7,119,88,139,779,146,895,119,19735,814,147182,300,0
9,1,SG12,P24_T3,SG12@P24_T3,en,de,T,3,P24,278,1180,29047,756199,845447,5,5,5,1145,88,146,857,154,981,2173,286175,1283,192365,300,0


## Step 6: Repeat Steps 4–5 with the segment (SG) tables (guided)

Now, repeat steps 4–5 with the segment (SG) tables. First you need to read them into a DataFrame that will be called `segments` (such creativity 🤯 😎 🤓 ).

In comparison to Step 5, only two things will change in our code calling the `read_TPRDB_tables()` function:

- the name of the variable we assign the function's output to
    - was `sessions` but will now be `segments`
- the value of the `extension` argument in the funciton
    - was `"ss"` but will now be `"sg"`

---

### **!!! ⚠️ Important ⚠️ !!!**

Once again, where we are running this notebook from matters:

#### **On the CRITT Jupyter server?**

If this is the case, then you should copy and paste the following code into the next cell:

```py
segments = read_TPRDB_tables(
    studies=["SG12"],
    extension="sg",
    mothership=True,
    user="PUBLIC",
    verbose=1,
)
```

---

#### **Are you somewhere else?**

In this case, you need to use your mothership clone you created before, so you should copy and paste the following code into the next cell (notice that we define the `path` of the mothership clone folder using the `mothership_clone_location` variable we declared in Step 3):

```py
segments = read_TPRDB_tables(
    studies=["SG12"],
    extension="sg",
    mothership=False,
    path=f"{mothership_clone_location}/tprdb-mothership-clone",
    user="PUBLIC",
    verbose=1,
)
```


In [ ]:
# 💻 Paste correct code from previous markdown cell depending on your situation, and then run the cell.


Reading: SG12	with 139 'sg' Tables
Total 'sg' data rows: 925, columns: 45


In [21]:
# First, check the segments DataFrame's shape
print(f"📊 The segments table has {segments.shape[0]} rows and {segments.shape[1]} columns")

📊 The segments table has 925 rows and 45 columns


In [ ]:
# Then, display the first few rows of the segments DataFrame with all columns visible
with pd.option_context("display.max_columns", None):
    display(segments.head())

,Id,Study,Session,StudySession,SL,TL,Task,Text,Part,KUI,PUB,STseg,TTseg,LenS,LenT,TokS,TokT,Ins,Del,Dur,FixS,TrtS,FixT,TrtT,Nedit,PreGap,PostGap,TB1000,TG1000,TD1000,TB_KUI,TG_KUI,TD_KUI,TB_PUB,TG_PUB,TD_PUB,TB_2PUB,TG_2PUB,TD_2PUB,TB_4PUB,TG_4PUB,TD_4PUB,String,HTot,HTotN
0,1,SG12,P02_T3,SG12@P02_T3,en,de,T,3,P02,278,1247,1,1,44,50,7,8,56,2,82963,305,69982,193,53060,1,115752,2308,8,70607,12356,20,76970,5993,6,68360,14603,4,64616,18347,3,60685,22278,Spielberg_zeigt_Peking_die_Rote_Karte_wegen_Da...,0.5252,0.1145
1,2,SG12,P02_T3,SG12@P02_T3,en,de,T,3,P02,278,1247,2,2,172,270,31,43,317,38,280147,472,101047,821,199884,2,2308,62836,22,210457,69690,80,239944,40203,21,209443,70704,17,202673,77474,12,185218,94929,"Es_ist_eine_Geste_,_durch_die_die_chinesische_...",1.0552,0.2301
2,3,SG12,P02_T3,SG12@P02_T3,en,de,T,3,P02,278,1247,3,3,201,227,37,33,306,44,258369,405,88257,801,142351,1,60403,12168,25,190634,67735,85,220544,37825,25,190634,67735,15,171446,86923,8,149137,109232,Sein_Rücktritt_steht_im_Zusammenhang_mit_den_a...,1.4410,0.3143
3,4,SG12,P02_T3,SG12@P02_T3,en,de,T,3,P02,278,1247,4,4,230,271,37,41,317,28,178373,657,139089,363,76925,2,49077,22074,24,112385,65988,80,140979,37394,21,109203,69170,11,91417,86956,4,64553,113820,"China_,_das_hohe_Investitionen_in_die_sudanesi...",0.7913,0.1726
4,5,SG12,P02_T3,SG12@P02_T3,en,de,T,3,P02,278,1247,5,5,220,256,34,40,276,16,85816,382,71093,334,89370,2,22074,58659,20,38862,53943,67,60481,32324,16,34602,58203,9,22075,70730,2,0,92805,"Obwohl_Spielberg_immer_wieder_hervorhebt_,_das...",0.8368,0.1825


## Step 7: Repeat Steps 4–5 with the source-token (ST) tables (self exploration)

You are on your own now 🥳🤘 In the following code cells, repeat Steps 4–5 with the source-token (ST) tables.

If you're hungry for more, don't worry: the next lesson, **[Module 01 Lesson 02: Study Summary Info](02_Summary_Info.ipynb)**, will delve into examining TPR-DB study summary information in greater depth.

In [ ]:
# You're the captain now 🛩️! Try reading the ST tables into a new DataFrame


In [ ]:
# You're the captain now 🛩️! Try getting the shape of the DataFrame you created


In [ ]:
# You're the captain now 🛩️! Try viewing the DataFrame with the options you prefer
